# Fast Intent Clustering (WildChat)

This notebook loads the **filtered English-only** dataset, samples a **configurable number** of conversations (default ~1000), and runs **fast** intent clustering:

1. **Rule-based**: keyword rules assign each first user message to one of a few big intent categories (coding, creative, commercial, education, support, casual/other)—no API, instant.
2. **Embedding + K-means**: first user messages are embedded with a local model (`sentence-transformers`), then clustered with K-means—no per-message LLM.
3. **Optional LLM summarization**: after clustering, the LLM is called **once per cluster** with a sample of messages to get a short description of what each cluster is about (K calls total, not thousands).

Data is loaded from `data/english_chunks/` if available (run the Export English-only section in `explore.ipynb` first); otherwise from Hugging Face with sampling.

In [1]:
# Configuration (edit and run first)
ENGLISH_CHUNKS_DIR = "data/english_chunks"
SAMPLE_N = 1000
RANDOM_SEED = 42
N_CLUSTERS = 8
TARGET_SUBCLUSTER_PCT = 0.05  # Each sub-cluster targets ~5% of total (within each intent)
USE_RULE_BASED = True
USE_EMBEDDING = True
USE_SUBCLUSTERS = True  # Embedding + K-means within each intent category
SUMMARIZE_WITH_LLM = True  # Set True to run one LLM call per cluster/subcategory (requires OPENAI_API_KEY)

## Load and prepare sample

We load the English-only dataset (from parquet chunks or Hugging Face), then sample N conversations and keep only rows with **non-empty first user message** so clustering is meaningful.

In [2]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

from eda_utils import load_english_chunked_parquet, sample_by_conversation
from intent_clustering import prepare_sample

load_dotenv()

chunks_dir = Path(ENGLISH_CHUNKS_DIR)
if chunks_dir.exists() and list(chunks_dir.glob("english_*.parquet")):
    df_full = load_english_chunked_parquet(chunks_dir)
    df = prepare_sample(df_full, n=SAMPLE_N, seed=RANDOM_SEED)
    print(f"Loaded from {chunks_dir}; after sampling and dropping empty text: {len(df)} rows.")
else:
    from datasets import load_dataset
    dataset = load_dataset("allenai/WildChat", split="train")
    sampled = sample_by_conversation(dataset, n=SAMPLE_N, pct=None, seed=RANDOM_SEED)
    df_raw = sampled.to_pandas()
    df = prepare_sample(df_raw, n=len(df_raw), seed=RANDOM_SEED)
    print(f"Loaded from Hugging Face; after dropping empty text: {len(df)} rows.")

print(f"Columns: {list(df.columns)}")
df.head(3)

Loaded from data/english_chunks; after sampling and dropping empty text: 1000 rows.
Columns: ['conversation_id', 'model', 'timestamp', 'conversation', 'turn', 'language', 'openai_moderation', 'detoxify_moderation', 'toxic', 'redacted', 'text']


,conversation_id,model,timestamp,conversation,turn,language,openai_moderation,detoxify_moderation,toxic,redacted,text
0,7c2ea6e4afa7dc22fa00fa974146b58f,gpt-3.5-turbo,2023-05-19 08:37:44+00:00,"[{'content': 'How to be like Nerd 1, Nerd 2 (S...",2,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.0002007092407438904, 'i...",False,False,"How to be like Nerd 1, Nerd 2 (SpongeBob Squar..."
1,4dfc2a6542e1f1be568462b45c214abc,gpt-4,2023-04-22 02:11:13+00:00,[{'content': 'fivem scripting I want to create...,4,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.00011475541396066546, '...",False,False,fivem scripting I want to create a basic dynam...
2,ddf52e99e2f8c46aecfd43e83343a6e6,gpt-3.5-turbo,2023-07-09 10:19:29+00:00,[{'content': '(We received it the Major case b...,1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.00011529319453984499, '...",False,False,(We received it the Major case by awaz today 0...


## Rule-based intent

Fast keyword rules assign each message to one of a few big intent categories. **Commercial-related** (product recommendations, comparisons, reviews, etc.) is checked first so messages that match both commercial and another category are labeled as commercial. No API calls.

In [3]:
from intent_clustering import assign_intent_rule_based

df["intent_rule"] = assign_intent_rule_based(df["text"])
print("Intent distribution (rule-based):")
display(df["intent_rule"].value_counts().to_frame("count"))

Intent distribution (rule-based):


,count
intent_rule,
casual_other,377
coding,292
creative_writing,203
education,46
commercial_product,46
support,36


## Sub-clusters within each category (embedding + K-means)

Within each rule-based intent category, we run embedding + K-means so that **each sub-cluster is about 5% of total data** (configurable via `TARGET_SUBCLUSTER_PCT`). So a large category gets more sub-clusters, a small one gets fewer. This yields finer-grained sub-categories without per-message LLM.

In [ ]:
from intent_clustering import add_subclusters_per_intent

if USE_SUBCLUSTERS and "intent_rule" in df.columns:
    df = add_subclusters_per_intent(
        df,
        intent_col="intent_rule",
        text_col="text",
        total_count=len(df),
        target_pct=TARGET_SUBCLUSTER_PCT,
        random_state=RANDOM_SEED,
        show_progress=True,
    )
    print("Sub-cluster counts per intent:")
    display(df.groupby(["intent_rule", "sub_cluster_id"]).size().unstack(fill_value=0))
else:
    print("USE_SUBCLUSTERS is False or no intent_rule; skipping per-category sub-clustering.")

## LLM name and description for each sub-category

For each sub-category (intent + sub_cluster_id), we call the LLM once to get a **short name** (1–3 words, like the category names) and a **one-sentence description**. Set `SUMMARIZE_WITH_LLM = True` to run.

In [ ]:
import textwrap

if SUMMARIZE_WITH_LLM and "sub_cluster_id" in df.columns:
    from intent_clustering import name_all_subcategories
    subcat_df = name_all_subcategories(
        df,
        intent_col="intent_rule",
        sub_cluster_col="sub_cluster_id",
        text_col="text",
        n_sample=15,
        seed=RANDOM_SEED,
    )
    width = 88
    for _, row in subcat_df.iterrows():
        intent, sid, count, name, desc = row["intent_rule"], row["sub_cluster_id"], row["count"], row["name"], (row["description"] or "")
        print(f"--- {intent} / sub_{sid} (n={count}) | {name} ---")
        print(textwrap.fill(desc, width=width))
        print()
    display(subcat_df)
else:
    print("SUMMARIZE_WITH_LLM is False or no sub_cluster_id; skipping LLM name/description.")

## Embedding + K-means clustering

We embed first user messages with a local model (`all-MiniLM-L6-v2`), then run K-means to get clusters. No API calls; runs in seconds to a minute on ~1K texts.

In [4]:
from intent_clustering import embed_texts, cluster_embeddings

if USE_EMBEDDING and len(df) > 0:
    embeddings = embed_texts(df["text"], show_progress=True)
    df["cluster_id"] = cluster_embeddings(
        embeddings, method="kmeans", n_clusters=N_CLUSTERS, random_state=RANDOM_SEED
    )
    print("Cluster sizes:")
    dist = df["cluster_id"].value_counts().sort_index()
    display(dist.to_frame("count").assign(pct=100.0 * dist.values / len(df)))
else:
    print("USE_EMBEDDING is False or no data; skipping embedding/clustering.")

/Users/Larry.Jin/miniconda3/envs/wildchat/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2306.08it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 16/16 [00:01<00:00, 12.08it/s]


Cluster sizes:


,count,pct
cluster_id,,
0,193,19.3
1,108,10.8
2,116,11.6
3,90,9.0
4,162,16.2
5,166,16.6
6,60,6.0
7,105,10.5


## Optional: LLM summarization per cluster

We call the LLM **once per cluster** with a sample of messages to get a short description of what each cluster is about. Set `SUMMARIZE_WITH_LLM = True` and add `OPENAI_API_KEY` to `.env` to run.

In [5]:
import textwrap

if SUMMARIZE_WITH_LLM and "cluster_id" in df.columns:
    from intent_clustering import summarize_all_clusters
    summary_df = summarize_all_clusters(
        df, cluster_col="cluster_id", text_col="text", n_sample=15, seed=RANDOM_SEED
    )
    width = 88
    for _, row in summary_df.iterrows():
        cid, count, summary = row["cluster_id"], row["count"], (row["summary"] or "")
        print(f"--- Cluster {cid} (n={count}) ---")
        print(textwrap.fill(summary, width=width))
        print()
    display(summary_df)
else:
    print("SUMMARIZE_WITH_LLM is False or no cluster_id column; skipping LLM summarization.")

--- Cluster 0 (n=193) ---
The common theme among these users is a request for assistance or information on a
variety of topics, ranging from personal correspondence and technical inquiries to
creative writing and game development. Users are typically seeking guidance,
clarification, or creative input to enhance their projects or communications.

--- Cluster 1 (n=108) ---
The common theme among these users is that they are requesting detailed image prompts
for the generative AI "Midjourney" based on specific concepts they provide. They are
looking for assistance in crafting prompts that will guide the AI in visualizing their
ideas effectively.

--- Cluster 2 (n=116) ---
The common theme among these messages is the request for creative writing prompts or
scenarios, often involving character development, dialogue, or specific narrative
situations. Users are typically asking for detailed stories, scripts, or scenes that
explore relationships, emotions, and imaginative concepts across vario

,cluster_id,count,summary
0,0,193,The common theme among these users is a reques...
1,1,108,The common theme among these users is that the...
2,2,116,The common theme among these messages is the r...
3,3,90,The common theme among these users is a reques...
4,4,162,The common theme among these users is a reques...
5,5,166,The common theme among these messages is that ...
6,6,60,The common theme among these messages is the e...
7,7,105,The common theme among these messages is a ble...


## Optional: commercial slice

Cross-tab of rule-based intent vs cluster (or count of commercial intent).

In [6]:
if "intent_rule" in df.columns:
    commercial_count = (df["intent_rule"] == "commercial_product").sum()
    print(f"Conversations with rule-based commercial intent: {commercial_count}")
if "intent_rule" in df.columns and "cluster_id" in df.columns:
    print("\nIntent vs cluster (counts):")
    display(pd.crosstab(df["intent_rule"], df["cluster_id"]))

Conversations with rule-based commercial intent: 46

Intent vs cluster (counts):


cluster_id,0,1,2,3,4,5,6,7
intent_rule,,,,,,,,
casual_other,109,0,15,53,75,71,18,36
coding,33,108,21,10,28,73,0,19
commercial_product,10,0,4,4,12,5,7,4
creative_writing,22,0,71,19,26,6,21,38
education,15,0,4,1,12,5,4,5
support,4,0,1,3,9,6,10,3
